# Exploratory Data Analysis

### Setup & Imports

In [ ]:
# Core scientific and geospatial libraries
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely import wkt
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical and Bayesian modeling tools
import arviz as az
import libpysal as lps
from esda.moran import Moran
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pymc as pm
import pytensor.tensor as pt

# Utilities and environment setup
from pathlib import Path
import warnings
import gc
import time
import datetime
import scipy as sp

# Suppress warnings for cleaner notebook presentation
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="viridis")

### Load & Parse Data

In [ ]:
DATA_PATH = Path("/home/.../final_integrated_dataset.csv") #change for the file path

# ── Global style constants ────────────────────────────────────────────────────
CMAP_SEQ = "YlOrRd"
REGION_ORDER = ["N", "NE", "SE", "S", "CO"]
REGION_COLORS = ["#004e98", "#e63946", "#f4a261", "#2a9d8f", "#a8dadc"]
REGION_MAP = {
    "AM": "N",
    "PA": "N",
    "RR": "N",
    "AP": "N",
    "TO": "N",
    "RO": "N",
    "AC": "N",
    "MA": "NE",
    "PI": "NE",
    "CE": "NE",
    "RN": "NE",
    "PB": "NE",
    "PE": "NE",
    "AL": "NE",
    "SE": "NE",
    "BA": "NE",
    "MG": "SE",
    "ES": "SE",
    "RJ": "SE",
    "SP": "SE",
    "PR": "S",
    "SC": "S",
    "RS": "S",
    "MS": "CO",
    "MT": "CO",
    "GO": "CO",
    "DF": "CO",
}
print("✅ Setup complete")

In [ ]:
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")

# Parse WKT geometry → GeoDataFrame
df_raw["_geom"] = gpd.GeoSeries.from_wkt(df_raw["geometry"])
gdf = gpd.GeoDataFrame(df_raw, geometry="_geom", crs="EPSG:4674")
gdf.drop(columns=["geometry"], inplace=True)
gdf.rename_geometry("geometry", inplace=True)

# Drop redundant merge-suffix columns
_DROP = [
    "NM_MUN_road",
    "SIGLA_UF_road",
    "area_km2",
    "municipality_name",
    "state_name",
    "municipality_name_embed",
    "state_name_embed",
    "municipality_id",
    "state_code",
    "extraction_date",
    "data_source",
]
gdf.drop(columns=[c for c in _DROP if c in gdf.columns], inplace=True)

gdf.rename(
    columns={
        "CD_MUN": "ibge_code",
        "NM_MUN": "municipality",
        "SIGLA_UF": "state",
        "AREA_KM2": "area_km2",
    },
    inplace=True,
)

# Derived columns (computed once so all subsequent cells always have them)
gdf["region"] = gdf["state"].map(REGION_MAP)
gdf["failure_rate"] = gdf["failed"] / (gdf["active"] + gdf["failed"])

# Business density metrics (per km²)
gdf["active_dens_km2"] = gdf["active"] / gdf["area_km2"]
gdf["failed_dens_km2"] = gdf["failed"] / gdf["area_km2"]
gdf["hq_dens_km2"] = gdf["active_head_offices"] / gdf["area_km2"]
gdf["branch_dens_km2"] = gdf["active_branches"] / gdf["area_km2"]

print(f"GeoDataFrame: {len(gdf):,} municipalities | CRS: {gdf.crs}")
gdf.head(3)

### Missing Value Audit

In [ ]:
num_cols = gdf.select_dtypes(include=np.number).columns.tolist()
missing = (gdf[num_cols].isna().mean() * 100).sort_values(ascending=False)
nz = missing[missing > 0]

fig, ax = plt.subplots(figsize=(10, max(4, len(nz) * 0.28)))
ax.barh(nz.index, nz.values, color=plt.cm.RdYlGn_r(nz.values / 100))
ax.axvline(20, color="red", ls="--", lw=0.9, label="20%")
ax.axvline(50, color="darkred", ls=":", lw=0.9, label="50%")
ax.set_xlabel("% Missing")
ax.set_title("Missing Values per Column", fontweight="bold")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f"Cols >50% missing: {(missing>50).sum()} | <5% missing: {(missing<5).sum()}")

### Brazil Overview Map

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))
gdf.plot(ax=ax, color="#2d6a4f", edgecolor="white", linewidth=0.08, alpha=0.85)
ax.set_title("Brazilian Municipalities — IBGE 2022", fontsize=14, fontweight="bold")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

### Road Infrastructure Analysis

#### How the data was collected and calculated

**Data source:** OpenStreetMap PBF file for Brazil (~4GB), downloaded from Geofabrik.

**Extraction (`extract_roads.py`):**
The `osmium` library parses the PBF file and extracts every OSM *way* tagged
with a `highway=*` key (motorway, trunk, primary, secondary, residential, …).
Each road segment is stored as a `LineString` with its geographic coordinates.

**Metric calculation (`calculate_road_metrics.py`):**
- `total_road_km` — sum of geodesic lengths of all road segments within each municipality polygon (spatial join via chunked GeoDataFrame operations).
- `road_density_km_km2` — `total_road_km / area_km2`. Measures how well-served a municipality is by road infrastructure per unit area.
- `intersection_count` — number of road junctions. Detected using an *Endpoint-First Topology Scan*: a two-pass algorithm that first counts start/end-point occurrences (Pass 1), then identifies T-junctions where interior points coincide with endpoints of other roads (Pass 2). This avoids loading the full coordinate graph into RAM.
- `intersection_density` — `intersection_count / area_km2`.
- `has_highway` — binary flag (1/0) indicating presence of a motorway or trunk road.

#### Key Results
- **High road density** → urbanised areas with dense street grids (São Paulo, Rio de Janeiro metro regions).
- **High intersection density** → complex, well-connected street networks (often correlates with GDP and population).
- **Low values in the North** → large Amazon municipalities with very sparse road coverage.

#### The National Veins
At the national level, the distribution of the **3.7 million km** road network perfectly mirrors Brazil's economic inequality. The median road density of **1.04 km/km²** masks a stark contrast: the Southeast (SE) and South (S) are dense, interconnected hubs, while the North (Amazon) remains an infrastructural frontier. For market entry, this road density acts as a ceiling on potential scale.

In [ ]:
ROAD_COLS = [
    c
    for c in [
        "total_road_km",
        "road_density_km_km2",
        "intersection_count",
        "intersection_density",
        "has_highway",
    ]
    if c in gdf.columns
]
print(
    f"Road data coverage: {gdf['road_density_km_km2'].notna().sum():,} / {len(gdf):,}"
)
gdf[ROAD_COLS].describe().round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 10))
p99 = gdf["road_density_km_km2"].quantile(0.99)
gdf.assign(_v=gdf["road_density_km_km2"].clip(upper=p99)).plot(
    column="_v",
    cmap=CMAP_SEQ,
    legend=True,
    legend_kwds={"label": "km/km² (clipped p99)", "shrink": 0.6},
    missing_kwds={"color": "#cccccc", "label": "No data"},
    ax=axes[0],
    edgecolor="none",
)
axes[0].set_title("Road Density (km/km²)", fontweight="bold")
axes[0].axis("off")

p99i = gdf["intersection_density"].quantile(0.99)
gdf.assign(_v=gdf["intersection_density"].clip(upper=p99i)).plot(
    column="_v",
    cmap="Purples",
    legend=True,
    legend_kwds={"label": "intersections/km² (clipped p99)", "shrink": 0.6},
    missing_kwds={"color": "#cccccc", "label": "No data"},
    ax=axes[1],
    edgecolor="none",
)
axes[1].set_title("Intersection Density (intersections/km²)", fontweight="bold")
axes[1].axis("off")
plt.suptitle(
    "Road Network — Spatial Distribution", fontsize=15, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
# Top 15 most road-dense municipalities
top15 = (
    gdf[["municipality", "state", "road_density_km_km2"]]
    .dropna()
    .nlargest(15, "road_density_km_km2")
)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(
    top15["municipality"] + " (" + top15["state"] + ")",
    top15["road_density_km_km2"],
    color="#2d6a4f",
)
ax.set_xlabel("Road Density (km/km²)")
ax.set_title("Top 15 — Road Density", fontweight="bold")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### Satellite Spectral Analysis

#### How the data was collected and calculated

**Platform:** Google Earth Engine (GEE), running server-side on Google's infrastructure.

**Image collection:** Sentinel-2 Level-2A Surface Reflectance (atmospherically corrected).
The script filters images for the target year, applies a cloud mask (removes pixels
with cloud probability > 20% using the `QA60` band), then takes a **median composite**
— i.e., for each pixel the median value across all valid observations is used.
This effectively removes cloud contamination and seasonal outliers.

**Spatial aggregation (`extract_embeddings.py`):**
For each municipality polygon, `ee.Image.reduceRegion()` computes the **mean** of every
band and index across all pixels within the polygon at 100m spatial resolution.

**Variables produced:**
| Variable | Formula | Meaning |
|----------|---------|---------|
| `red`, `green`, `blue`, `nir`, `swir_1/2` | raw band values | Surface reflectance (0–1 scale) |
| `ndvi` | (NIR − Red) / (NIR + Red) | Vegetation density. Range −1 to 1; high values = dense green vegetation |
| `evi` | 2.5 × (NIR−Red) / (NIR+6×Red−7.5×Blue+1) | Enhanced vegetation, less soil-noise than NDVI |
| `savi` | 1.5 × (NIR−Red) / (NIR+Red+0.5) | Soil-Adjusted VI; better in sparse vegetation |
| `ndwi` | (Green − NIR) / (Green + NIR) | Water surfaces; positive values = water |
| `mndwi` | (Green − SWIR1) / (Green + SWIR1) | Modified water index; more sensitive than NDWI |
| `ndbi` | (SWIR1 − NIR) / (SWIR1 + NIR) | Built-up index; positive values = urban/bare surfaces |
| `urban_index` | custom | Urban land-use proxy |

#### Key Results
- **High NDVI** in the North (Amazon) and South (Atlantic Forest — PR, SC, RS).
- **High NDBI** in the São Paulo / Rio de Janeiro metro regions.
- **Negative NDWI/MNDWI** in arid Northeast municipalities (Caatinga biome).

#### The View from Above
Satellite intelligence allows us to see beyond official records. The strong negative correlation (~ -0.8) between **NDVI** (Greenness) and **NDBI** (Built-up Index) provides a high-fidelity map of the "Concrete Jungle" effect. This spectral data acts as a proxy for both consumer concentration and operational costs associated with urban density.

In [ ]:
SPECTRAL = [
    c
    for c in ["ndvi", "evi", "savi", "ndwi", "mndwi", "ndbi", "urban_index"]
    if c in gdf.columns
]
top_states = gdf["state"].value_counts().head(10).index.tolist()
sdf = gdf[gdf["state"].isin(top_states)]

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()
for i, idx in enumerate(SPECTRAL):
    ax = axes[i]
    grps = [sdf.loc[sdf["state"] == s, idx].dropna().values for s in top_states]
    bp = ax.boxplot(grps, labels=top_states, patch_artist=True, showfliers=False)
    for patch, c in zip(bp["boxes"], plt.cm.tab10.colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.7)
    ax.set_title(idx.upper(), fontweight="bold")
    ax.tick_params(axis="x", rotation=45, labelsize=7)
axes[-1].set_visible(False)
plt.suptitle(
    "Spectral Indices by State (Top 10)", fontsize=13, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 10))
gdf.plot(
    column="ndvi",
    cmap="YlGn",
    legend=True,
    legend_kwds={"label": "NDVI", "shrink": 0.6},
    missing_kwds={"color": "#dddddd"},
    vmin=-0.1,
    vmax=0.8,
    ax=axes[0],
    edgecolor="none",
)
axes[0].set_title("NDVI — Vegetation", fontweight="bold")
axes[0].axis("off")

gdf.plot(
    column="ndbi",
    cmap="RdPu",
    legend=True,
    legend_kwds={"label": "NDBI", "shrink": 0.6},
    missing_kwds={"color": "#dddddd"},
    ax=axes[1],
    edgecolor="none",
)
axes[1].set_title("NDBI — Built-up Index", fontweight="bold")
axes[1].axis("off")
plt.suptitle(
    "Satellite Indices — Spatial Distribution", fontsize=15, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.show()

### Socioeconomic Indicators

#### How the data was collected and calculated

**Sources:**
- `GDP_per_capita` — IBGE National Accounts (PIB dos Municípios), value in BRL.
- `avrg_monthly_salary` — RAIS (Relação Anual de Informações Sociais), average formal salary.
- `HDI_income`, `HDI_longevity`, `HDI_educational` — PNUD Brazil Municipal HDI components (0–1 scale).
- `population_density` — IBGE Census estimate (inhabitants / km²).
- `n_bank_branches` — Banco Central count of licensed bank branches per municipality.
- `vehicle_fleet_2021` — DENATRAN vehicle registration count.

**Collection (`preprocess_cities.py`):**
Raw Excel files from each institution are loaded, columns are renamed to a
unified schema, municipality IBGE codes are standardised to 7 digits, and all
sources are merged on `ibge_code`.

#### Key Results
- GDP and HDI are strongly correlated with road density (urban municipalities are richer).
- The Southeast (SE) region dominates economic indicators; the North (N) and Northeast (NE) lag significantly.
- Bank branches echo economic development — municipalities with zero branches are mostly rural.

In [ ]:
ECO = [
    c
    for c in [
        "GDP_per_capita",
        "avrg_monthly_salary",
        "population_density",
        "HDI_income",
        "HDI_longevity",
        "HDI_educational",
        "n_bank_branches",
        "vehicle_fleet_2021",
    ]
    if c in gdf.columns
]
gdf[ECO].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 10))
p95 = gdf["GDP_per_capita"].quantile(0.95)
gdf.assign(_v=gdf["GDP_per_capita"].clip(upper=p95)).plot(
    column="_v",
    cmap="plasma",
    legend=True,
    legend_kwds={"label": "BRL/capita (p95 clipped)", "shrink": 0.6},
    missing_kwds={"color": "#dddddd"},
    ax=axes[0],
    edgecolor="none",
)
axes[0].set_title("GDP per Capita", fontweight="bold")
axes[0].axis("off")
gdf.plot(
    column="HDI_income",
    cmap="viridis",
    legend=True,
    legend_kwds={"label": "HDI Income", "shrink": 0.6},
    missing_kwds={"color": "#dddddd"},
    ax=axes[1],
    edgecolor="none",
)
axes[1].set_title("HDI Income", fontweight="bold")
axes[1].axis("off")
plt.suptitle(
    "Socioeconomic — Spatial Distribution", fontsize=15, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.ticker as mtick

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
groups = [
    gdf.loc[gdf["region"] == r, "GDP_per_capita"].dropna().values for r in REGION_ORDER
]
bp = axes[0].boxplot(groups, labels=REGION_ORDER, patch_artist=True, showfliers=False)
for patch, c in zip(bp["boxes"], REGION_COLORS):
    patch.set_facecolor(c)
    patch.set_alpha(0.8)
axes[0].set_title("GDP per Capita by Region", fontweight="bold")
axes[0].set_ylabel("BRL")
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
sal = gdf.groupby("state")["avrg_monthly_salary"].median().sort_values()
axes[1].barh(sal.index, sal.values, color="#457b9d")
axes[1].set_title("Median Monthly Salary by State", fontweight="bold")
axes[1].set_xlabel("BRL")
plt.tight_layout()
plt.show()

### Business Activity — City, State & Regional Analysis

#### How the data was collected and calculated

**Source:** Receita Federal do Brasil — CNPJ public dataset.
Each record is a legal entity (empresa) with address, CNAE economic activity code,
and status (`ACTIVE` / `CLOSED` / `SUSPENDED`) - I filtered only those that declared bankruptcy as the reason or were suspended for failing to meet their obligations - in the last five years.

**Collection & aggregation (`preprocess_cities.py`):**
Records are filtered to establishments with addresses in Brazil, then grouped by
municipality IBGE code. For each municipality the following counts are computed:

| Variable | Meaning |
|----------|---------|
| `active` | Total active establishments (firm status = ACTIVE) |
| `failed` | Closed/deregistered establishments (status = CLOSED OR SUSPENDED) |
| `active_head_offices` | Active establishments classified as *Matriz* (HQ) |
| `active_branches` | Active establishments classified as *Filial* (branches) |
| `failure_rate` | `failed / (active + failed)` — proportion of closed firms |

#### Key Results
- **Active firms** concentrate strongly in São Paulo, Rio de Janeiro, and Belo Horizonte.
- **Failure rate** is surprisingly similar across regions (most municipalities 30–45%).
- **Business Density** reveals that while total numbers are high in capitals, the *intensity* of business activity (firms/km²) is extremely peaked in metropolitan cores.
- **Head Office Density:** Highly clustered in major financial centers (SP, RJ), indicating centralised decision-making.
- **Branch Density:** More spatially dispersed than HQs, following consumer markets and logistics hubs.
- **State-level** aggregation reveals SP's economic dominance; NE states have fewer firms per capita.
- **Regional** radar chart shows clear SE advantage in total firms, but Northeast has competitive firm density in some coastal cities.

#### The Pulse of Enterprise
Despite the massive disparity in economic volume, with **São Paulo alone** housing over **5.5 million active firms**, the **national failure rate of 35.1%** is relatively stable across regions. This suggests that while wealth increases the *number* of players, the *risk* of failure is a structural constant that requires localized spatial intelligence to navigate effectively.

In [ ]:
biz_cols = [
    c
    for c in [
        "active",
        "failed",
        "active_head_offices",
        "active_branches",
        "failed_head_offices",
        "failed_branches",
        "active_dens_km2",
        "hq_dens_km2",
    ]
    if c in gdf.columns
]
print(
    f"Municipalities with business data: {gdf['active'].notna().sum():,} / {len(gdf):,}"
)
gdf[biz_cols].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 9))

p95_act = gdf["active"].quantile(0.95)
gdf.assign(_v=gdf["active"].clip(upper=p95_act)).plot(
    column="_v",
    cmap="Blues",
    legend=True,
    legend_kwds={"label": "# active firms (p95 clip)", "shrink": 0.6},
    missing_kwds={"color": "#eeeeee"},
    ax=axes[0],
    edgecolor="none",
)
axes[0].set_title("Active Firms (Count)", fontweight="bold")
axes[0].axis("off")

# New: Business Density Map
p99_dens = gdf["active_dens_km2"].quantile(0.99)
gdf.assign(_v=gdf["active_dens_km2"].clip(upper=p99_dens)).plot(
    column="_v",
    cmap="cividis",
    legend=True,
    legend_kwds={"label": "Firms / km² (p99 clip)", "shrink": 0.6},
    missing_kwds={"color": "#eeeeee"},
    ax=axes[1],
    edgecolor="none",
)
axes[1].set_title("Business Density (Firms/km²)", fontweight="bold")
axes[1].axis("off")

if "active_head_offices" in gdf.columns:
    p95_ho = gdf["active_head_offices"].quantile(0.95)
    gdf.assign(_v=gdf["active_head_offices"].clip(upper=p95_ho)).plot(
        column="_v",
        cmap="Greens",
        legend=True,
        legend_kwds={"label": "Active HQ (p95 clip)", "shrink": 0.6},
        missing_kwds={"color": "#eeeeee"},
        ax=axes[2],
        edgecolor="none",
    )
    axes[2].set_title("Active Head Offices (Count)", fontweight="bold")
    axes[2].axis("off")

plt.suptitle("Business Activity — Municipality Level", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# HQs per km2
p99_hq = gdf["hq_dens_km2"].quantile(0.99)
gdf.assign(_v=gdf["hq_dens_km2"].clip(upper=p99_hq)).plot(
    column="_v",
    cmap="Purples",
    legend=True,
    legend_kwds={"label": "HQs / km²", "shrink": 0.6},
    missing_kwds={"color": "#eeeeee"},
    ax=axes[0],
    edgecolor="none",
)
axes[0].set_title("Head Office Density", fontweight="bold")
axes[0].axis("off")

# Branches per km2
p99_br = gdf["branch_dens_km2"].quantile(0.99)
gdf.assign(_v=gdf["branch_dens_km2"].clip(upper=p99_br)).plot(
    column="_v",
    cmap="Oranges",
    legend=True,
    legend_kwds={"label": "Branches / km²", "shrink": 0.6},
    missing_kwds={"color": "#eeeeee"},
    ax=axes[1],
    edgecolor="none",
)
axes[1].set_title("Branch Density", fontweight="bold")
axes[1].axis("off")

plt.suptitle("Business Structure — Density Maps", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
state_biz = (
    gdf.groupby("state", observed=True)
    .agg(
        total_active=pd.NamedAgg("active", "sum"),
        total_failed=pd.NamedAgg("failed", "sum"),
        mean_failure=pd.NamedAgg("failure_rate", "mean"),
        total_hq=pd.NamedAgg("active_head_offices", "sum"),
        n_municipalities=pd.NamedAgg("municipality", "count"),
    )
    .reset_index()
)
state_biz["active_per_muni"] = state_biz["total_active"] / state_biz["n_municipalities"]

# Build a state-level GeoDataFrame by dissolving municipality polygons
state_gdf = (
    gdf[["state", "geometry"]]
    .dissolve(by="state", aggfunc="first")
    .reset_index()
    .merge(state_biz, on="state")
)
state_gdf = gpd.GeoDataFrame(state_gdf, geometry="geometry", crs=gdf.crs)

fig, axes = plt.subplots(1, 3, figsize=(24, 9))

# Total active firms
state_gdf.plot(
    column="total_active",
    cmap="Blues",
    legend=True,
    legend_kwds={"label": "Total active firms", "shrink": 0.6},
    ax=axes[0],
    edgecolor="white",
    linewidth=0.5,
)
for _, row in state_gdf.iterrows():
    c = row.geometry.centroid
    axes[0].text(
        c.x, c.y, row["state"], ha="center", va="center", fontsize=6, color="black"
    )
axes[0].set_title("Total Active Firms by State", fontweight="bold")
axes[0].axis("off")

# Mean failure rate
state_gdf.plot(
    column="mean_failure",
    cmap="Reds",
    legend=True,
    legend_kwds={"label": "Mean failure rate", "shrink": 0.6},
    vmin=0,
    vmax=0.8,
    ax=axes[1],
    edgecolor="white",
    linewidth=0.5,
)
for _, row in state_gdf.iterrows():
    c = row.geometry.centroid
    axes[1].text(
        c.x, c.y, f'{row["mean_failure"]:.2f}', ha="center", va="center", fontsize=6
    )
axes[1].set_title("Mean Business Failure Rate by State", fontweight="bold")
axes[1].axis("off")

# Active firms per municipality within state
state_gdf.plot(
    column="active_per_muni",
    cmap="YlGn",
    legend=True,
    legend_kwds={"label": "Active firms / municipality", "shrink": 0.6},
    ax=axes[2],
    edgecolor="white",
    linewidth=0.5,
)
for _, row in state_gdf.iterrows():
    c = row.geometry.centroid
    axes[2].text(c.x, c.y, row["state"], ha="center", va="center", fontsize=6)
axes[2].set_title("Active Firms per Municipality (within State)", fontweight="bold")
axes[2].axis("off")

plt.suptitle("Business Activity — State Level", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sb = state_biz.sort_values("total_active", ascending=True)
axes[0].barh(sb["state"], sb["total_active"], color="#1d3557")
axes[0].set_title("Total Active Firms by State", fontweight="bold")
axes[0].set_xlabel("Count")
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))

sb2 = state_biz.sort_values("mean_failure", ascending=True)
colors = [
    REGION_COLORS[REGION_ORDER.index(REGION_MAP.get(s, "N"))] for s in sb2["state"]
]
axes[1].barh(sb2["state"], sb2["mean_failure"], color=colors)
axes[1].set_title("Mean Business Failure Rate by State", fontweight="bold")
axes[1].set_xlabel("Rate")
patches = [
    mpatches.Patch(color=c, label=r) for c, r in zip(REGION_COLORS, REGION_ORDER)
]
axes[1].legend(handles=patches, fontsize=8, title="Region")
plt.tight_layout()
plt.show()

In [ ]:
region_biz = (
    gdf[gdf["region"].isin(REGION_ORDER)]
    .groupby("region", observed=True)
    .agg(
        n_municipalities=pd.NamedAgg("municipality", "count"),
        total_active=pd.NamedAgg("active", "sum"),
        total_failed=pd.NamedAgg("failed", "sum"),
        mean_failure_rate=pd.NamedAgg("failure_rate", "mean"),
        total_hq=pd.NamedAgg("active_head_offices", "sum"),
        total_branches=pd.NamedAgg("active_branches", "sum"),
    )
    .reindex(REGION_ORDER)
)
region_biz["active_per_muni"] = (
    region_biz["total_active"] / region_biz["n_municipalities"]
)
region_biz["hq_share"] = region_biz["total_hq"] / region_biz["total_active"]
print(region_biz.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Total active vs failed
x = np.arange(len(REGION_ORDER))
w = 0.35
axes[0].bar(x - w / 2, region_biz["total_active"], w, label="Active", color="#2d6a4f")
axes[0].bar(x + w / 2, region_biz["total_failed"], w, label="Closed", color="#e63946")
axes[0].set_xticks(x)
axes[0].set_xticklabels(REGION_ORDER)
axes[0].set_title("Active vs Closed Firms by Region", fontweight="bold")
axes[0].legend()
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

# HQ vs Branches composition
axes[1].bar(REGION_ORDER, region_biz["total_hq"], label="Head Offices", color="#457b9d")
axes[1].bar(
    REGION_ORDER,
    region_biz["total_branches"],
    bottom=region_biz["total_hq"],
    label="Branches",
    color="#a8dadc",
)
axes[1].set_title("HQ vs Branch Composition by Region", fontweight="bold")
axes[1].legend()
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))

# Failure rate + active per municipality
ax2 = axes[2].twinx()
axes[2].bar(
    REGION_ORDER,
    region_biz["mean_failure_rate"],
    color="#e63946",
    alpha=0.7,
    label="Failure rate",
)
ax2.plot(
    REGION_ORDER,
    region_biz["active_per_muni"],
    "o-",
    color="#1d3557",
    lw=2,
    label="Active/municipality",
)
axes[2].set_ylabel("Mean Failure Rate", color="#e63946")
ax2.set_ylabel("Active firms / municipality", color="#1d3557")
axes[2].set_title("Failure Rate & Firm Density by Region", fontweight="bold")
axes[2].legend(loc="upper left")
ax2.legend(loc="upper right")
plt.suptitle(
    "Business Activity — Regional Level", fontsize=14, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.show()

In [ ]:
region_gdf = (
    gdf[gdf["region"].isin(REGION_ORDER)][["region", "geometry"]]
    .dissolve(by="region", aggfunc="first")
    .reset_index()
    .merge(region_biz.reset_index(), on="region")
)
region_gdf = gpd.GeoDataFrame(region_gdf, geometry="geometry", crs=gdf.crs)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
region_gdf.plot(
    column="total_active",
    cmap="Blues",
    legend=True,
    legend_kwds={"label": "Total active firms", "shrink": 0.6},
    ax=axes[0],
    edgecolor="white",
    linewidth=1,
)
for _, row in region_gdf.iterrows():
    c = row.geometry.centroid
    axes[0].text(
        c.x,
        c.y,
        f'{row["region"]}\n{row["total_active"]/1e6:.1f}M',
        ha="center",
        va="center",
        fontsize=9,
        fontweight="bold",
    )
axes[0].set_title("Total Active Firms by Region", fontweight="bold")
axes[0].axis("off")

region_gdf.plot(
    column="mean_failure_rate",
    cmap="Reds",
    legend=True,
    legend_kwds={"label": "Mean failure rate", "shrink": 0.6},
    vmin=0,
    vmax=0.8,
    ax=axes[1],
    edgecolor="white",
    linewidth=1,
)
for _, row in region_gdf.iterrows():
    c = row.geometry.centroid
    axes[1].text(
        c.x,
        c.y,
        f'{row["region"]}\n{row["mean_failure_rate"]:.2f}',
        ha="center",
        va="center",
        fontsize=9,
        fontweight="bold",
    )
axes[1].set_title("Mean Business Failure Rate by Region", fontweight="bold")
axes[1].axis("off")
plt.suptitle("Business Activity — Regional Level", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

### Correlation Analysis

#### How the data was calculated

**Method:** Pearson correlation coefficient ($r$).
The correlation matrix quantifies the linear relationship between pairs of numeric variables.
- $r = 1$: Perfect positive correlation
- $r = -1$: Perfect negative correlation
- $r = 0$: No linear correlation

**Variables included:**
Economic (GDP, Salary, Active Firms), Infrastructure (Road density, Intersections), and Satellite (NDVI, NDBI) metrics.

#### Key Results
- **Road Density vs GDP ($r > 0.6$):** Strong positive correlation. Places with more roads tend to be wealthier.
- **NDVI vs NDBI ($r \approx -0.8$):** Strong negative correlation. Areas with dense vegetation (NDVI) have few buildings (NDBI), and vice versa.
- **Active Firms vs Population:** Naturally highly correlated, as firms concentrate in population centers.

In [ ]:
CORR_WANT = [
    "road_density_km_km2",
    "intersection_density",
    "ndvi",
    "ndbi",
    "urban_index",
    "GDP_per_capita",
    "avrg_monthly_salary",
    "population_density",
    "HDI_income",
    "HDI_longevity",
    "HDI_educational",
    "active",
    "failure_rate",
    "n_bank_branches",
    "vehicle_fleet_2021",
]
CORR_COLS = [c for c in CORR_WANT if c in gdf.columns]
# Compute Pearson correlation to identify linear associations and multi-collinearity
corr = gdf[CORR_COLS].corr()

fig, ax = plt.subplots(figsize=(14, 12))
# Custom Heatmap Visualization with significant correlation labels
im = ax.imshow(corr, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, shrink=0.8, label="Pearson r")
ax.set_xticks(range(len(CORR_COLS)))
ax.set_yticks(range(len(CORR_COLS)))
ax.set_xticklabels(CORR_COLS, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(CORR_COLS, fontsize=8)
for i in range(len(CORR_COLS)):
    for j in range(len(CORR_COLS)):
        v = corr.iloc[i, j]
        if abs(v) > 0.3:
            ax.text(
                j,
                i,
                f"{v:.2f}",
                ha="center",
                va="center",
                fontsize=6,
                color="white" if abs(v) > 0.7 else "black",
            )
ax.set_title("Correlation Matrix — Key Indicators", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### Cross-Domain Scatter Analysis

#### Purpose
Scatter plots reveal non-linear relationships and clusters that a simple correlation coefficient might miss.
We explore relationships between **Infrastructure, Economy, and Environment**.

#### Key Pairs Analyzed
1.  **Road Density vs GDP/capita:** Is infrastructure a prerequisite for wealth? (Log-linear relationship often observed).
2.  **NDVI vs GDP/capita:** Do richer cities tend to have less green cover? (The "concrete jungle" effect).
3.  **NDBI vs Active Firms:** Interpreting NDBI as a proxy for urbanisation intensity vs business activity.

**Trend lines:** Plotting a log-linear regression trend (dashed red line) to visualise the general direction of the relationship.
**Colors:** Points are coloured by **Region** to highlight spatial clustering (e.g., North/Amazon vs Southeast/Industrial).

In [ ]:
pairs = [
    ("road_density_km_km2", "GDP_per_capita", "Road Density vs GDP/capita"),
    ("road_density_km_km2", "avrg_monthly_salary", "Road Density vs Avg Salary"),
    ("ndvi", "GDP_per_capita", "NDVI vs GDP/capita"),
    ("ndbi", "active", "NDBI vs Active Firms"),
    ("population_density", "road_density_km_km2", "Pop Density vs Road Density"),
    ("HDI_income", "road_density_km_km2", "HDI Income vs Road Density"),
]
pairs = [(x, y, t) for x, y, t in pairs if x in gdf.columns and y in gdf.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for ax, (x, y, title) in zip(axes, pairs):
    sub = gdf[[x, y, "region"]].dropna()
    for (reg, grp), color in zip(sub.groupby("region"), REGION_COLORS):
        ax.scatter(grp[x], grp[y], alpha=0.2, s=5, color=color, label=reg)
    xs_r, ys_r = sub[x].clip(lower=0), sub[y].clip(lower=0)
    if xs_r.std() > 0 and ys_r.std() > 0:
        m, b = np.polyfit(np.log1p(xs_r), np.log1p(ys_r), 1)
        xl = np.linspace(xs_r.quantile(0.05), xs_r.quantile(0.95), 100)
        ax.plot(xl, np.expm1(m * np.log1p(xl) + b), "r--", lw=1.2, label="trend")
    ax.set_title(title, fontweight="bold", fontsize=9)
    ax.set_xlabel(x, fontsize=8)
    ax.set_ylabel(y, fontsize=8)
    ax.legend(fontsize=6)
plt.suptitle("Cross-Domain Scatter Analysis", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### Green Areas & Quality of Life Analysis

#### Hypothesis
Does the presence of green areas (NDVI) correlate with better quality of life (HDI, GDP) and infrastructure (Roads)?

**Variables:**
- **Green Presence:** `ndvi` (Vegetation Index).
- **Quality of Life:** `HDI_educational`, `HDI_longevity`, `GDP_per_capita`.
- **Infrastructure:** `road_density_km_km2`.

**Key Results:**
- **Quality of Life (Health):** There is a moderate positive correlation ($0.385$) between NDVI and longevity (HDI_longevity). This suggests that areas with more vegetation tend to have better health outcomes or longer life expectancies.
- **Quality of Life (Economic):** There is a weak positive correlation ($0.165$) between NDVI and GDP per capita. While there is a slight upward trend, higher economic wealth does not strongly guarantee or require high vegetation density in this dataset.
- **Infrastructure:** There is no significant correlation ($-0.022$) between NDVI and road density. The presence of green space appears to be independent of how dense the road network is, indicating that infrastructure development does not necessarily come at the cost of green space in all observed areas.

In [ ]:
QOL_COLS = ["ndvi", "GDP_per_capita", "road_density_km_km2", "HDI_longevity"]
# Filter for columns present in dataframe
curr_qol = [c for c in QOL_COLS if c in gdf.columns]

if len(curr_qol) > 1:
    pd.plotting.scatter_matrix(
        gdf[curr_qol],
        alpha=0.2,
        figsize=(12, 12),
        diagonal="kde",
        color="#2a9d8f",
        grid=True,
    )
    plt.suptitle(
        "Green Areas (NDVI) vs Quality of Life Indicators",
        fontsize=14,
        fontweight="bold",
        y=0.92,
    )
    plt.show()

    # Text Correlation Report
    print("Correlation with NDVI:")
    print(gdf[curr_qol].corr()["ndvi"].sort_values(ascending=False).to_string())

### Regional Aggregation Summary

#### Methodology
Data is aggregated by the five official macro-regions of Brazil:
- **N (North):** Amazon rainforest, expansive municipalities, lower density.
- **NE (Northeast):** Coastal density, arid interior (Caatinga), mixed economic performance.
- **SE (Southeast):** Industrial heartland (SP, RJ, MG), highest GDP and infrastructure density.
- **S (South):** High HDI, balanced agricultural/industrial economy.
- **CO (Center-West):** Agricultural powerhouse (Soy/Cattle), high GDP/capita but lower population density.

**Visualization: Regional Radar Chart**
To compare regions on a single chart, we normalize key indicators to a 0–1 scale:
$$ x_{norm} = \\frac{x - \\min(x)}{\\max(x) - \\min(x)} $$
This allows us to see the "shape" of each region's profile (e.g., is it "Rich but Sparse" or "Poor but Dense"?).

In [ ]:
region_full = (
    gdf[gdf["region"].isin(REGION_ORDER)]
    .groupby("region", observed=True)
    .agg(
        n_municipalities=pd.NamedAgg("municipality", "count"),
        total_area_km2=pd.NamedAgg("area_km2", "sum"),
        population=pd.NamedAgg("resident_population", "sum"),
        median_gdp_pc=pd.NamedAgg("GDP_per_capita", "median"),
        median_road_den=pd.NamedAgg("road_density_km_km2", "median"),
        mean_ndvi=pd.NamedAgg("ndvi", "mean"),
        mean_ndbi=pd.NamedAgg("ndbi", "mean"),
        total_active=pd.NamedAgg("active", "sum"),
        mean_failure=pd.NamedAgg("failure_rate", "mean"),
    )
    .reindex(REGION_ORDER)
)
print(region_full.to_string())

In [ ]:
import sys, math, warnings

RCOLS = [
    c
    for c in ["median_gdp_pc", "median_road_den", "mean_ndvi", "total_active"]
    if c in region_full.columns
]
RLABELS = {
    "median_gdp_pc": "GDP/capita",
    "median_road_den": "Road Density",
    "mean_ndvi": "NDVI",
    "total_active": "Active Firms",
}

normed = region_full[RCOLS].copy()
normed = (normed - normed.min()) / (normed.max() - normed.min() + 1e-9)
N = len(RCOLS)
angles = [n / N * 2 * math.pi for n in range(N)] + [0]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True})
for (region, row), color in zip(normed.iterrows(), REGION_COLORS):
    vals = row.tolist() + [row.tolist()[0]]
    ax.plot(angles, vals, "o-", lw=2, label=region, color=color)
    ax.fill(angles, vals, alpha=0.07, color=color)
ax.set_xticks(angles[:-1])
ax.set_xticklabels([RLABELS.get(c, c) for c in RCOLS], fontsize=10)
ax.set_title("Regional Profile Radar (normalised 0–1)", fontweight="bold", pad=15)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))
plt.tight_layout()
plt.show()

#### Regional disparities (Radar Chart)

- The South (S): Shows the most balanced and high-performing profile, leading in NDVI (greenness), GDP/capita, and Road Density.
- The Southeast (SE): While it has high road density and the highest concentration of Active Firms, its NDVI is significantly lower than that of the South.
- The North (N): Possesses high NDVI but demonstrates the lowest levels of infrastructure (Road Density) and economic activity (Active Firms and GDP/capita) in this comparison.
- The Northeast (NE) & Center-West (CO): These regions show more moderate levels across all indicators, with the Northeast specifically showing very low GDP/capita despite having some road infrastructure.

### Key Findings Summary

This summary is generated from the latest dataset state.
It highlights:

- **Data Coverage:** Percentage of municipalities with valid data for key layers (Roads, Satellite, Economy).
- **National Averages:** Benchmarks for road density, vegetation index, and failure rates.
- **Economic Leaders:** Identifying the top performing states by business volume.

This summary serves as a quick health-check of the integrated dataset before proceeding to modeling.

### Conclusion
Our journey through the data brings us to a clear conclusion: 
1. **Space is the Variable**: Success in one municipality is a strong signal for its neighbors.
2. **Infrastructure is the Floor**: You cannot have high-volume market entry without a dense road network.
3. **Intelligence is the Advantage**: By combining Bayesian uncertainty (Investment Risk) with empirical data, we can identify "Safe Opportunities" that aggregate indicators would otherwise miss.

*The next phase is to expand the São Paulo pilot logic into a unified national prediction engine.*

In [ ]:
def sp(label, value):
    print(f"  {label:<42}: {value}")


print("=" * 65)
print("  iGuide Project — EDA Summary")
print("=" * 65)
sp("Total municipalities", f"{len(gdf):,}")
sp("States covered", gdf["state"].nunique())
print()
print("  🛣️  Road Network")
rd = gdf["road_density_km_km2"]
sp("Coverage", f"{rd.notna().sum():,} / {len(gdf):,} ({rd.notna().mean()*100:.1f}%)")
sp("Median road density", f"{rd.median():.4f} km/km²")
if "total_road_km" in gdf.columns:
    sp("Total road km", f"{gdf['total_road_km'].sum():,.0f} km")
print()
print("  🛰️  Satellite")
sp(
    "NDVI coverage",
    f"{gdf['ndvi'].notna().sum():,} ({gdf['ndvi'].notna().mean()*100:.1f}%)",
)
sp("Mean NDVI", f"{gdf['ndvi'].mean():.3f}")
sp("Mean NDBI", f"{gdf['ndbi'].mean():.3f}")
print()
print("  🏛️  Economy")
gdp = gdf["GDP_per_capita"]
sp("GDP coverage", f"{gdp.notna().sum():,} ({gdp.notna().mean()*100:.1f}%)")
sp("Median GDP per capita", f"BRL {gdp.median():,.0f}")
print()
print("  🏢  Business Activity")
sp("Total active firms (BR)", f"{gdf['active'].sum():,.0f}")
sp("Total closed firms (BR)", f"{gdf['failed'].sum():,.0f}")
sp("National mean failure rate", f"{gdf['failure_rate'].mean():.3f}")
print()
print("  Top 3 States by Active Firms:")
top3 = state_biz.nlargest(3, "total_active")[["state", "total_active"]]
for _, r in top3.iterrows():
    sp(f"    {r['state']}", f"{r['total_active']:,.0f}")
print("=" * 65)